# ConvNeXt M3 — QAT đúng 1 epoch từ checkpoint Kaggle
Notebook thử nghiệm tách biệt: tải `fp32_best.pt` từ checkpoint dataset version 2, calibration 512 ảnh, train một frozen-observer QAT epoch, convert INT8, benchmark FP32–INT8 và upload kết quả vào private dataset riêng. Không ghi đè pipeline 30/11 epoch.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Bật GPU và Internet trong Kaggle Settings'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', torch.cuda.get_device_properties(0).total_memory / 2**30)

## 1. Clone project và cài dependencies

In [ ]:
REPO = Path('/kaggle/working/EchteAI')
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/NguyenDucThang-tb/EchteAI.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[coco]', 'psutil', 'kagglehub'], check=True)
import kagglehub
print('Repository:', Path.cwd())

## 2. Thông số
Chỉ thay username nếu cần.

In [ ]:
KAGGLE_USERNAME = 'nguyenducthangtb'
FP32_DATASET = f'{KAGGLE_USERNAME}/echteai-seadronessee-m3-checkpoints/versions/2'
RESULT_DATASET = f'{KAGGLE_USERNAME}/echteai-seadronessee-m3-qat-one-epoch'
SEADRONESSEE_DATASET = 'ubiratanfilho/sds-dataset'
VARIANT = 'M3'
print('FP32 source:', FP32_DATASET)
print('Private result dataset:', RESULT_DATASET)

## 3. Tải dataset và FP32 checkpoint version 2

In [ ]:
def find_dataset_root(start):
    start = Path(start)
    for candidate in [start, *[p.parent.parent for p in start.rglob('instances_train.json')]]:
        if (candidate/'annotations/instances_train.json').exists() and (candidate/'images/train').is_dir():
            return candidate
    raise FileNotFoundError(f'Không tìm thấy SeaDronesSee trong {start}')

DATA_ROOT = find_dataset_root(kagglehub.dataset_download(SEADRONESSEE_DATASET))
FP32_DOWNLOAD = Path(kagglehub.dataset_download(FP32_DATASET, force_download=True))
matches = list(FP32_DOWNLOAD.rglob('fp32_best.pt'))
assert matches, f'Không tìm thấy fp32_best.pt trong {FP32_DOWNLOAD}'
FP32_CHECKPOINT = matches[0]
payload = torch.load(FP32_CHECKPOINT, map_location='cpu', weights_only=False)
print('FP32 checkpoint:', FP32_CHECKPOINT)
print('Source epoch:', payload.get('epoch'))
print('Source metrics:', payload.get('metrics', {}))
print('Size MB:', FP32_CHECKPOINT.stat().st_size/2**20)

## 4. Tạo runtime config QAT một epoch

In [ ]:
import yaml
config = yaml.safe_load(Path('configs/seadronessee_colab.yaml').read_text())
OUTPUT = Path('/kaggle/working/qat_one_epoch_m3')
OUTPUT.mkdir(parents=True, exist_ok=True)
config['dataset'].update({
    'train_images': str(DATA_ROOT/'images/train'), 'train_annotations': str(DATA_ROOT/'annotations/instances_train.json'),
    'val_images': str(DATA_ROOT/'images/val'), 'val_annotations': str(DATA_ROOT/'annotations/instances_val.json'),
    'test_images': str(DATA_ROOT/'images/val'), 'test_annotations': str(DATA_ROOT/'annotations/instances_val.json'),
})
config['training']['qat_epochs'] = 1
config['training']['qat_batch_size'] = 1
config['quantization']['weight_only_warmup_epochs'] = 0
config['quantization']['observer_freeze_epochs'] = 1
config['quantization']['calibration_images'] = 512
config['quantization']['phase_learning_rates']['frozen'] = 1e-5
config['output'] = {
    'directory': str(OUTPUT), 'fp32_best': str(FP32_CHECKPOINT), 'fp32_last': str(FP32_CHECKPOINT),
    'qat_best': str(OUTPUT/'qat_best.pt'), 'qat_last': str(OUTPUT/'qat_last.pt'),
    'int8_model': str(OUTPUT/'selective_int8.pt'), 'evaluation_json': str(OUTPUT/'evaluation.json'),
    'benchmark_json': str(OUTPUT/'benchmark.json'), 'epoch_benchmarks': str(OUTPUT/'epoch_benchmarks.json'),
}
RUNTIME_CONFIG = Path('/kaggle/working/qat_one_epoch_runtime.yaml')
RUNTIME_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False))
print('Runtime config:', RUNTIME_CONFIG)

## 5. Calibration + train QAT đúng 1 epoch + convert INT8

In [ ]:
command = [sys.executable, '-u', 'scripts/train_qat.py', '--config', str(RUNTIME_CONFIG), '--fp32-checkpoint', str(FP32_CHECKPOINT), '--variant', VARIANT, '--epochs-this-run', '1']
print('Command:', ' '.join(command), flush=True)
log_path = OUTPUT/'qat_train.log'
env = os.environ.copy(); env['PYTHONUNBUFFERED'] = '1'
print('Persistent log:', log_path, flush=True)
with log_path.open('a', encoding='utf-8') as log_file:
    log_file.write('\n===== QAT ONE EPOCH START =====\n'); log_file.flush()
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in process.stdout:
        print(line, end='', flush=True); log_file.write(line); log_file.flush()
    code = process.wait()
if code: raise subprocess.CalledProcessError(code, command)

## 6. Benchmark FP32–INT8 công bằng trên CPU và upload kết quả

In [ ]:
assert (OUTPUT/'selective_int8.pt').exists(), 'INT8 checkpoint chưa được tạo'
subprocess.run([
    sys.executable, '-u', 'scripts/compare_fp32_int8.py', '--config', str(RUNTIME_CONFIG),
    '--fp32-checkpoint', str(FP32_CHECKPOINT), '--int8-checkpoint', str(OUTPUT/'selective_int8.pt'),
    '--images', '100', '--threads', '1', '--output', str(OUTPUT/'fp32_int8_comparison.json'),
], check=True)
for path in sorted(OUTPUT.glob('*')): print(f'{path.name:30s} {path.stat().st_size/2**20:9.2f} MB')
kagglehub.dataset_upload(RESULT_DATASET, str(OUTPUT), version_notes='M3 QAT one epoch from FP32 checkpoint dataset version 2')
print('Private result dataset uploaded:', RESULT_DATASET)